# EDA: Zalo Photos Assignment 2026

**Mục tiêu:** Khám phá dữ liệu toàn diện trước khi training, phát hiện các vấn đề chất lượng dữ liệu.

**Hai index embedding được dùng song song:**
- **SigLIP2** (`google/siglip2-so400m-patch16-384`) — semantic index chính: UMAP, text search, similarity, class contamination.
- **C-RADIOv4** (`nv_labs/c-radio_v4-h`) — visual QA index phụ: near-duplicates, leaky splits, kiểm tra cluster thị giác.

**Plan:**
1. Setup & clone dataset
2. Dataset Inventory & Sanity Checks
3. SigLIP2 Semantic Index (similarity + visualization + zero-shot)
4. C-RADIOv4 Visual QA Index (near-dups + leaky splits + uniqueness)
5. FiftyOne App
6. Phát hiện quan trọng từ EDA

In [ ]:
from pathlib import Path

import fiftyone as fo

import z_photos
from z_photos.conf import Config
from z_photos.datasets import build_dataset

cfg = Config(
    data_root=Path("../data"),
    model_name_or_path="google/siglip2-so400m-patch16-384",
    seed=42,
)

dataset = build_dataset(z_photos.__name__, bronze_dir=cfg.bronze_dir)

EDA_DATASET_NAME = "z_photos_eda"


def get_or_clone_eda(source_dataset, name: str) -> fo.Dataset:
    """Clone source dataset for EDA, delete existing clone if present."""
    if name in fo.list_datasets():
        fo.delete_dataset(name)
    return source_dataset.clone(name=name, persistent=False)


eda = get_or_clone_eda(dataset, EDA_DATASET_NAME)
print(eda)

## Section 2 — Dataset Inventory & Sanity Checks

Kiểm tra toàn diện trước khi EDA:
1. **Class distribution** — mất cân bằng giữa train/test?
2. **Metadata sanity** — kích thước ảnh, file corrupt?
3. **Exact duplicates** — file trùng lặp chính xác (hash).

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd

# ── 1. Compute metadata (width, height, size_bytes)
eda.compute_metadata()

# ── 2. Class distribution per split
train_view = eda.match_tags("train")
test_view = eda.match_tags("test")

train_counts = Counter(train_view.values("ground_truth.label"))
test_counts = Counter(test_view.values("ground_truth.label"))

all_classes = sorted(set(train_counts) | set(test_counts))
df_dist = pd.DataFrame(
    {
        "class": all_classes,
        "train": [train_counts.get(c, 0) for c in all_classes],
        "test": [test_counts.get(c, 0) for c in all_classes],
    }
).set_index("class")
df_dist["total"] = df_dist["train"] + df_dist["test"]
df_dist["train_pct"] = (df_dist["train"] / df_dist["train"].sum() * 100).round(1)
df_dist["test_pct"] = (df_dist["test"] / df_dist["test"].sum() * 100).round(1)

print("=== Class Distribution ===")
display(df_dist.style.format({"train_pct": "{:.1f}%", "test_pct": "{:.1f}%"}))
print(f"\nTrain total: {df_dist['train'].sum()} | Test total: {df_dist['test'].sum()}")

In [ ]:
widths = eda.values("metadata.width")
heights = eda.values("metadata.height")
sizes = eda.values("metadata.size_bytes")

missing_meta = sum(1 for w in widths if w is None)
print(f"Samples với metadata thiếu (có thể corrupt): {missing_meta}")

w_arr = np.array([w for w in widths if w], dtype=float)
h_arr = np.array([h for h in heights if h], dtype=float)
s_arr = np.array([s for s in sizes if s], dtype=float)
ar_arr = w_arr / h_arr

df_meta = pd.DataFrame(
    {
        "metric": ["Width (px)", "Height (px)", "Aspect ratio", "File size (KB)"],
        "min": [w_arr.min(), h_arr.min(), ar_arr.min(), s_arr.min() / 1024],
        "max": [w_arr.max(), h_arr.max(), ar_arr.max(), s_arr.max() / 1024],
        "mean": [w_arr.mean(), h_arr.mean(), ar_arr.mean(), s_arr.mean() / 1024],
        "std": [w_arr.std(), h_arr.std(), ar_arr.std(), s_arr.std() / 1024],
    }
).set_index("metric")
display(df_meta.style.format("{:.1f}"))

### Exact Duplicates

`fob.compute_exact_duplicates` — so sánh hash file MD5/SHA, phát hiện file trùng chính xác. Không cần model.

In [ ]:
import fiftyone.brain as fob

# Hash-based, không cần model
exact_dups = fob.compute_exact_duplicates(eda, progress=False)

total_dup_samples = sum(len(v) for v in exact_dups.values())
print(f"Exact duplicate groups: {len(exact_dups)}")
print(f"Exact duplicate samples (to remove): {total_dup_samples}")

if exact_dups:
    for rep_id, dup_ids in list(exact_dups.items())[:5]:
        rep = eda[rep_id]
        fname = rep.filepath.split("/")[-1]
        print(f"\n  Representative [{rep.tags}] {rep.ground_truth.label}: .../{fname}")
        for did in dup_ids:
            s = eda[did]
            fname = s.filepath.split("/")[-1]
            print(f"    Duplicate [{s.tags}] {s.ground_truth.label}: .../{fname}")

## Section 3 — SigLIP2 Semantic Index (chính)

SigLIP2 là **zero-shot image-text model** → embedding có khả năng embed text, cho phép:
- `fob.compute_similarity` → text search trong App
- `fob.compute_visualization` → UMAP với semantic meaning
- `sort_by_similarity("a photo of X")` → kiểm tra class contamination

**Cách dùng:** FiftyOne HuggingFace Integration, load qua Zoo với `zero-shot-classification-transformer-torch`.

In [ ]:
import fiftyone.zoo as foz

# Load SigLIP2 qua FiftyOne HuggingFace Integration
# SigLIP2 là zero-shot model → hỗ trợ text embedding → text search trong App
siglip2_model = foz.load_zoo_model(
    "zero-shot-classification-transformer-torch",
    name_or_path="google/siglip2-so400m-patch16-384",
)
print(f"SigLIP2 loaded | can_embed_prompts={siglip2_model.can_embed_prompts}")

### SigLIP2 Similarity Index + UMAP Visualization

In [ ]:
import numpy as np
import torch
from PIL import Image
from transformers import AutoModelForZeroShotImageClassification, AutoProcessor

MODEL_ID = "google/siglip2-so400m-patch16-384"

# Load SigLIP2 as zero-shot model để có text_embeds + image_embeds đúng cách
siglip2_hf = AutoModelForZeroShotImageClassification.from_pretrained(MODEL_ID)
siglip2_hf.eval()
siglip2_proc = AutoProcessor.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
siglip2_hf = siglip2_hf.to(device)
print(f"SigLIP2 loaded on {device}")


def extract_siglip2_image_embeds(dataset, batch_size: int = 32) -> list[np.ndarray]:
    """Extract SigLIP2 L2-normalized image embeddings (image_embeds)."""
    all_embeddings = []
    filepaths = dataset.values("filepath")
    for i in range(0, len(filepaths), batch_size):
        batch_paths = filepaths[i : i + batch_size]
        images = [Image.open(p).convert("RGB") for p in batch_paths]
        # Pass dummy text to get image_embeds (SigLIP2 requires paired inputs)
        inputs = siglip2_proc(
            text=["placeholder"] * len(images),
            images=images,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
        ).to(device)
        with torch.no_grad():
            outputs = siglip2_hf(**inputs)
        # image_embeds: L2-normalized [B, 1152]
        embs = outputs.image_embeds.cpu().float().numpy()
        all_embeddings.extend(embs)
        print(f"  {min(i + batch_size, len(filepaths))}/{len(filepaths)}", end="\r")
    print()
    return all_embeddings


embeddings = extract_siglip2_image_embeds(eda, batch_size=16)
eda.set_values("siglip2_embeddings", embeddings)
print(f"SigLIP2 embeddings: {len(embeddings)} samples, dim={embeddings[0].shape[0]}")

In [ ]:
import fiftyone.utils.transformers as fout

# Wrap the raw HuggingFace SigLIP2 model for FiftyOne
# convert_transformers_model làm cho FiftyOne biết cách extract embeddings đúng cách
siglip2_fo = fout.convert_transformers_model(hf_model)
print(
    f"FO-wrapped SigLIP2: has_embeddings={siglip2_fo.has_embeddings}, can_embed_prompts={siglip2_fo.can_embed_prompts}"
)

# Similarity index: build từ pre-computed embeddings (sklearn backend ok)
siglip2_sim = fob.compute_similarity(
    eda,
    embeddings="siglip2_embeddings",
    brain_key="siglip2_sim",
    backend="sklearn",
    metric="cosine",
)
print(f"SigLIP2 similarity index ok. supports_prompts={siglip2_sim.supports_prompts}")

# UMAP visualization
fob.compute_visualization(
    eda,
    embeddings="siglip2_embeddings",
    method="umap",
    brain_key="siglip2_viz",
    num_dims=2,
    seed=cfg.seed,
)
print("SigLIP2 UMAP visualization ok.")

### SigLIP2 Zero-shot Classification (reference vs ground truth)

Dùng SigLIP2 để zero-shot predict → so sánh với `ground_truth` → lọc ngay ảnh nghi ngờ.

In [ ]:
import fiftyone.zoo as foz

# Classes từ ground_truth field (English names)
CLASSES = sorted(dataset.distinct("ground_truth.label"))
print(f"Classes: {CLASSES}")

# Load SigLIP2 zero-shot với classes đúng
siglip2_zs = foz.load_zoo_model(
    "zero-shot-classification-transformer-torch",
    name_or_path="google/siglip2-so400m-patch16-384",
    classes=CLASSES,
)
print(f"Zero-shot model classes: {siglip2_zs.classes}")

eda.apply_model(
    siglip2_zs,
    label_field="siglip2_pred",
    batch_size=32,
)

# Thống kê agreement
gt_labels = eda.values("ground_truth.label")
pred_labels = eda.values("siglip2_pred.label")

agree = sum(g == p for g, p in zip(gt_labels, pred_labels))
total = len(eda)
print(
    f"\nSigLIP2 zero-shot vs ground_truth agreement: {agree}/{total} = {agree / total * 100:.1f}%"
)

# Disagreements theo class
disagree_by_class: dict[str, list[str]] = {}
for g, p in zip(gt_labels, pred_labels):
    if g != p:
        disagree_by_class.setdefault(g, []).append(p)

rows = []
for gt_cls in sorted(disagree_by_class):
    preds = Counter(disagree_by_class[gt_cls])
    for pred_cls, cnt in preds.most_common():
        rows.append({"ground_truth": gt_cls, "siglip2_pred": pred_cls, "count": cnt})

df_disagree = pd.DataFrame(rows)
if not df_disagree.empty:
    print("\nSigLIP2 disagreements (ground_truth ≠ prediction):")
    display(df_disagree.style.hide(axis="index"))

### SigLIP2 Text Search: Kiểm tra Class Contamination

Tìm K-nearest neighbors theo text prompt → phát hiện ảnh không thuộc đúng class của chúng.

In [ ]:
from transformers import AutoTokenizer

# Text search với SigLIP2: dùng text_embeds từ zero-shot model
# text_embeds đã được L2-normalize → dot product = cosine similarity
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

text_queries = [
    "people celebrating Tet holiday in Vietnam",
    "a beach or seaside",
    "trees and nature landscape",
    "a park with people gathering",
    "a baby child playing",
    "mountain trekking hiking",
]


def embed_text_siglip2(query: str) -> np.ndarray:
    """Extract SigLIP2 L2-normalized text embedding."""
    # SigLIP2 cần paired image+text → dùng dummy image
    dummy_img = Image.open(eda.first().filepath).convert("RGB")
    inputs = siglip2_proc(
        text=[query],
        images=[dummy_img],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
    ).to(device)
    with torch.no_grad():
        outputs = siglip2_hf(**inputs)
    return outputs.text_embeds[0].cpu().float().numpy()


# Pre-loaded image embeddings (already L2-normalized)
emb_matrix = np.stack(eda.values("siglip2_embeddings"))  # [N, 1152]
gt_labels = eda.values("ground_truth.label")

rows = []
for query in text_queries:
    txt_emb = embed_text_siglip2(query)  # L2-normalized
    sims = emb_matrix @ txt_emb  # dot product = cosine similarity
    top5_idx = np.argsort(sims)[::-1][:5]
    top5_labels = [gt_labels[i] for i in top5_idx]
    top5_sims = [float(sims[i]) for i in top5_idx]
    rows.append(
        {
            "query": query,
            "top_1_label": top5_labels[0],
            "top_1_sim": f"{top5_sims[0]:.3f}",
            "top_5_labels": dict(Counter(top5_labels).most_common(3)),
        }
    )

df_search = pd.DataFrame(rows)
display(
    df_search[["query", "top_1_label", "top_1_sim", "top_5_labels"]].style.hide(
        axis="index"
    )
)

## Section 4 — C-RADIOv4 Visual QA Index (phụ)

C-RADIOv4 là **multi-teacher distilled model** (SigLIP2 + DINOv3 + SAM3) → embedding rất mạnh về visual detail.

Dùng cho:
- `compute_near_duplicates` — ảnh **rất giống nhau về thị giác** (crop, resize, re-composition)
- `compute_leaky_splits` — phát hiện data leakage train → test
- `compute_uniqueness` — phân tích cụm thị giác

**Cài đặt plugin (chạy 1 lần):**
```python
import fiftyone.zoo as foz
foz.register_zoo_model_source("https://github.com/harpreetsahota204/CRADIOv4")
```

In [ ]:
# Register C-RADIOv4 plugin source (chỉ cần chạy 1 lần)
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/CRADIOv4",
)

# Load C-RADIOv4-SO400M (nhanh hơn H, đủ cho dataset nhỏ này)
radio_model = foz.load_zoo_model("nv_labs/c-radio_v4-so400m")

# Compute embeddings vào field riêng
# num_workers=0 để tránh multiprocessing pickling error trên macOS Python 3.13
eda.compute_embeddings(
    radio_model,
    embeddings_field="radio_embeddings",
    batch_size=16,
    num_workers=0,
)
print("C-RADIOv4 embeddings computed.")

### C-RADIOv4: Near Duplicates

In [ ]:
# Build similarity index từ RADIO embeddings (dùng để near-dups + leaky splits)
radio_sim = fob.compute_similarity(
    eda,
    embeddings="radio_embeddings",
    brain_key="radio_sim",
    backend="sklearn",
    metric="cosine",
)

# Near duplicates: ảnh rất giống nhau về mặt thị giác
near_dup_index = fob.compute_near_duplicates(
    eda,
    similarity_index=radio_sim,
    threshold=0.1,  # cosine distance < 0.1 → very similar visually
)

dup_ids = near_dup_index.duplicate_ids
print(f"Near duplicate samples (C-RADIOv4, threshold=0.1): {len(dup_ids)}")
rate = len(dup_ids) / len(eda) * 100
print(f"Near-dup rate: {len(dup_ids)}/{len(eda)} = {rate:.1f}%")

if dup_ids:
    dup_samples = eda.select(dup_ids)
    dup_label_counts = Counter(dup_samples.values("ground_truth.label"))
    dup_tag_counts = Counter([t for ts in dup_samples.values("tags") for t in ts])
    df_nd = pd.DataFrame(
        {
            "class": list(dup_label_counts.keys()),
            "near_dup_count": list(dup_label_counts.values()),
        }
    ).set_index("class")
    display(df_nd.style.format("{:d}"))

### C-RADIOv4: Leaky Splits Detection

Tìm ảnh train và test quá giống nhau về nội dung thị giác → **data leakage** → metric bị inflate.  
Dataset nhỏ (~261 train / 60 test) → vài ảnh trùng có thể làm accuracy tăng giả tạo.

In [ ]:
# Tái dùng radio_sim index để detect leaky splits
leaky_index = fob.compute_leaky_splits(
    eda,
    splits=["train", "test"],
    similarity_index=radio_sim,
)

leaks_view = leaky_index.leaks_view()
print(f"Leaky samples (train↔test, C-RADIOv4): {len(leaks_view)}")

if len(leaks_view) > 0:
    print("\nCác samples bị leak:")
    for sample in leaks_view.iter_samples():
        fname = sample.filepath.split("/")[-1]
        print(f"  [{sample.tags}] {sample.ground_truth.label}: .../{fname}")
    # Tag leaks để review sau
    leaky_index.tag_leaks(tag="leaky")
    print("\n✓ Đã tag leaky samples với tag 'leaky'.")
else:
    print("✓ Không phát hiện leaky splits — train/test tách biệt tốt.")

### C-RADIOv4: Uniqueness & Representativeness

In [ ]:
# Uniqueness: tìm ảnh redundant (thấp) và outlier OOD (cao) dựa trên RADIO embeddings
fob.compute_uniqueness(
    eda,
    similarity_index=radio_sim,
    uniqueness_field="radio_uniqueness",
)

uniqueness_vals = eda.values("radio_uniqueness")
u_arr = np.array([v for v in uniqueness_vals if v is not None])
print(
    f"Uniqueness stats (C-RADIOv4): min={u_arr.min():.3f}  max={u_arr.max():.3f}  mean={u_arr.mean():.3f}"
)

# Top-5 most unique (potential OOD)
print("\nTop-5 most unique (potential outliers/OOD):")
for s in eda.sort_by("radio_uniqueness", reverse=True).limit(5).iter_samples():
    fname = s.filepath.split("/")[-1]
    print(
        f"  u={s.radio_uniqueness:.3f}  [{s.tags}]  {s.ground_truth.label}: .../{fname}"
    )

# Top-5 least unique (most redundant)
print("\nTop-5 least unique (most redundant):")
for s in eda.sort_by("radio_uniqueness", reverse=False).limit(5).iter_samples():
    fname = s.filepath.split("/")[-1]
    print(
        f"  u={s.radio_uniqueness:.3f}  [{s.tags}]  {s.ground_truth.label}: .../{fname}"
    )

# Representativeness: tìm ảnh "đại diện" nhất cho mỗi cluster
fob.compute_representativeness(
    eda,
    embeddings="radio_embeddings",
    representativeness_field="radio_representativeness",
    method="cluster-center",
)
print("\nRepresentativeness computed (radio_representativeness).")

## Section 5 — FiftyOne App

Mở App để:
- **Sidebar**: xem phân phối class, filter theo split/class, xem metadata
- **Embeddings Panel**: chọn `siglip2_viz` (UMAP, color by `ground_truth.label`) → xem clustering semantic
- **Sort by** `radio_uniqueness` → review outlier/OOD
- **Sort by** `siglip2_pred.label ≠ ground_truth.label` → review noise candidates
- **Text search**: nhập prompt vào search bar (dùng `siglip2_sim`) → `"beach seaside"`, `"tet holiday"`
- **Tag**: click ảnh → nhấn `t` → nhập tag `"noisy"`, `"ambiguous"`, `"leaky"`

In [ ]:
session = fo.launch_app(eda, auto=False)
print("FiftyOne App: http://localhost:5151")

## Section 6 — Phát hiện quan trọng từ EDA

### 6.1 Dataset Overview

| Split | Total | Classes |
|---|---|---|
| Train | 261 | baby_playing 39, gathering 24, lunar_new_year 32, nature 60, other 66, trekking 40 |
| Test | 60 | baby_playing 9, gathering 5, lunar_new_year 7, nature 14, other 16, trekking 9 |

**Class imbalance:** `other` (25.3%) vs `gathering` (9.2%) — tỷ lệ 2.75:1. Không cực đoan nhưng cần `class_weight` khi train.

---

### 6.2 Exact Duplicates — 14 groups (14 samples cần xóa)

**Phát hiện nghiêm trọng:**
- **8 ảnh `lunar_new_year` bị duplicate sang `other`** — cùng file, khác label. Đây là **nhãn nhiễu cố ý**: ảnh Tết được gán cả 2 class.
- **1 ảnh `baby_playing` ↔ `other`** — cùng file, khác class.
- **2 ảnh `lunar_new_year` ↔ `gathering`** — cùng file, class overlap có thể chấp nhận được.
- **2 ảnh `train` ↔ `test` exact duplicate** (`other`→`baby_playing`, `gathering`→`lunar_new_year`) — **data leakage nghiêm trọng**.

> **Hành động:** Dedup trước khi train. Ưu tiên remove bản `other` khi bản kia có class cụ thể.

---

### 6.3 Near Duplicates (C-RADIOv4, threshold=0.1) — 17 samples (5.3%)

| Class | Near-dup count |
|---|---|
| other | 9 |
| nature | 3 |
| gathering | 2 |
| lunar_new_year | 2 |
| baby_playing | 1 |

**By split:** train=14, test=3. `other` class có density cao nhất → nhiều ảnh rất giống nhau trong `other`.

---

### 6.4 Leaky Splits (C-RADIOv4) — 20 samples bị leak

**Nghiêm trọng nhất:** 20/321 samples (6.2%) có train↔test similarity cao.

Các cặp đáng chú ý:
- `train/other` → `test/baby_playing`: FILE GIỐNG HỆT (exact dup + label conflict)
- `train/gathering` → `test/lunar_new_year`: FILE GIỐNG HỆT
- `train/nature` ↔ `test/nature`: 4 cặp ảnh rất giống nhau (crop/resize)
- `train/trekking` ↔ `test/trekking`: 1 cặp

> **Hành động:** Dùng `no_leaks_view()` để tạo split sạch. Loại leaky samples khỏi evaluation metric.

---

### 6.5 SigLIP2 Zero-shot Agreement — 12.1% (39/321)

**Degenerate result:** SigLIP2 zero-shot phân loại **mọi ảnh là `lunar_new_year`** (100% accuracy cho class đó, 0% cho các class khác).

**Nguyên nhân:**
1. Label names (`baby_playing`, `trekking`, ...) thiếu ngữ cảnh văn hóa Việt Nam trong embedding space SigLIP2.
2. `lunar_new_year` có label text gần nhất với nội dung ảnh (hầu hết ảnh là cảnh sinh hoạt Việt Nam).
3. **Kết luận quan trọng:** Embedding space SigLIP2 không phân biệt tốt 6 class này bằng text labels. Cần **visual embeddings** (không phải zero-shot text) để phân loại.

---

### 6.6 SigLIP2 Text Search Results

| Query | Top-1 label | Top-5 labels |
|---|---|---|
| people celebrating Tet | lunar_new_year | gathering × 3, lunar_new_year × 2 |
| beach or seaside | (mixed) | other × 2, nature × 2 |
| trees and nature landscape | nature | nature × 5 ✓ |
| park with people gathering | (mixed) | other × 3, lunar_new_year × 2 |
| baby child playing | baby_playing | baby_playing × 5 ✓ |
| mountain trekking hiking | (mixed) | other × 3, lunar_new_year × 1, trekking × 1 |

**Phát hiện:**
- `nature` và `baby_playing` có **semantic cluster rõ ràng** → SigLIP2 phân tách tốt.
- `trekking` bị **lẫn với `other`** → class này không có visual uniqueness cao.
- `gathering` và `lunar_new_year` **overlap mạnh** → ranh giới mờ nhạt (cảnh tụ họp ↔ cảnh Tết).
- `beach/seaside` → top results là `other` và `nature` → **`other` chứa ảnh biển/ngoài trời** không thuộc 5 class còn lại.

---

### 6.7 C-RADIOv4 Uniqueness

- `other` class **chiếm TOP-5 most unique** (u=1.0 đến u=0.96) → class này là catch-all, nhiều OOD samples.
- `nature` **chiếm TOP-5 least unique** (u=0.17) → nhiều ảnh thiên nhiên rất giống nhau (cùng scene, khác crop).
- Least unique sample có tag `leaky` → xác nhận near-dup ↔ leaky correlation.

---

### 6.8 Tóm tắt hành động

| Vấn đề | Mức độ | Hành động |
|---|---|---|
| Exact dup cross-class (lunar↔other) | 🔴 Cao | Dedup, giữ class cụ thể |
| Exact dup train↔test | 🔴 Cao | Loại khỏi train hoặc test |
| Leaky splits (20 samples) | 🔴 Cao | Tag "leaky", exclude khỏi eval clean |
| Near-dups trong `other` | 🟡 Trung bình | Giảm weight hoặc dedup |
| gathering ↔ lunar_new_year overlap | 🟡 Trung bình | Augment, label smoothing |
| trekking bị confuse với other | 🟡 Trung bình | Thêm descriptive prompts |
| nature redundant (u thấp) | 🟢 Thấp | Không cần xử lý đặc biệt |
| other class OOD (u cao) | 🟢 Thấp | Confidence threshold khi inference |